# Postprocessing PureCLIP peaks without KO input control

## Input/output configuration

Edit these paths before running the analysis. Relative paths are resolved from the repository root. Run the cells in order using the R kernel.

In [ ]:
params <- list(
  plus_bw = "results/preprocessing/akna/crosslinked_nucleotides/replicates.plus.bw",
  minus_bw = "results/preprocessing/akna/crosslinked_nucleotides/replicates.minus.bw",
  pureclip_sites = "results/preprocessing/akna/peak_calling/replicates_noinputcontroldata_PureCLIP.crosslink_sites_short.bed",
  output_bed = "results/analysis/01_pureclip_postprocessing/replicates_noKO_curated_peaks.bed",
  merged_sites_bed = "results/analysis/01_pureclip_postprocessing/replicates_noKO_sign_sites_max_gap_width_7.bed"
)

project_root <- normalizePath(getwd(), mustWork = TRUE)
if (!file.exists(file.path(project_root, "workflow", "Snakefile"))) {
  project_root <- normalizePath(file.path(project_root, ".."), mustWork = TRUE)
}
if (!file.exists(file.path(project_root, "workflow", "Snakefile"))) {
  stop("Open this notebook from the repository root or its analysis directory.")
}
resolve_repo_path <- function(path) {
  if (grepl("^(/|[A-Za-z]:[/\\\\])", path)) path else file.path(project_root, path)
}
paths <- lapply(params, resolve_repo_path)


This notebook applies the PureCLIP postprocessing used for the manuscript. It
merges nearby sites, centers 9-nt windows on crosslink maxima, and splits wider
regions into non-overlapping 9-nt windows.

In [ ]:
plus_bw_file <- paths$plus_bw
minus_bw_file <- paths$minus_bw
peak_file <- paths$pureclip_sites
output_file <- paths$output_bed
sign_sites <- paths$merged_sites_bed

dir.create(dirname(output_file), recursive = TRUE, showWarnings = FALSE)
dir.create(dirname(sign_sites), recursive = TRUE, showWarnings = FALSE)


In [ ]:
library(GenomicRanges)
library(rtracklayer)
library(ggplot2)


In [ ]:
plus_cov <- import.bw(plus_bw_file, as = "Rle")
minus_cov <- abs(import.bw(minus_bw_file, as = "Rle"))
peaks <- import(peak_file)
cat("Number of peaks from PureCLIP:", length(peaks), "\n")


In [ ]:
peaks_sc <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
cat("Number of standard chromosome peaks:", length(peaks_sc), "\n")

peaks_sc_m <- reduce(peaks_sc, min.gapwidth = 8)
cat("Number of peaks after merging gaps up to 7 nt:", length(peaks_sc_m), "\n")
export(peaks_sc_m, sign_sites)

width_threshold <- 0
peaks_sc_m_pw <- peaks_sc_m[width(peaks_sc_m) > width_threshold]


In [ ]:
peaks_sc_m_pw_short <- peaks_sc_m_pw[width(peaks_sc_m_pw) <= 9]
peaks_sc_m_pw_short_plus <- peaks_sc_m_pw_short[
  strand(peaks_sc_m_pw_short) == "+"
]
max_pos_plus <- sapply(plus_cov[peaks_sc_m_pw_short_plus], which.max)
peaks_sc_m_pw_short_plus_centered <- GRanges(
  seqnames = seqnames(peaks_sc_m_pw_short_plus),
  ranges = IRanges(
    start = start(peaks_sc_m_pw_short_plus) + max_pos_plus - 1,
    width = 1
  ),
  strand = "+"
)

peaks_sc_m_pw_short_minus <- peaks_sc_m_pw_short[
  strand(peaks_sc_m_pw_short) == "-"
]
max_pos_minus <- sapply(
  lapply(minus_cov[peaks_sc_m_pw_short_minus], rev),
  which.max
)
peaks_sc_m_pw_short_minus_centered <- GRanges(
  seqnames = seqnames(peaks_sc_m_pw_short_minus),
  ranges = IRanges(
    start = end(peaks_sc_m_pw_short_minus) - max_pos_minus + 1,
    width = 1
  ),
  strand = "-"
)

peaks_sc_m_pw_sc <- c(
  peaks_sc_m_pw_short_plus_centered + 4,
  peaks_sc_m_pw_short_minus_centered + 4
)
cat("Number of resized short peaks:", length(peaks_sc_m_pw_sc), "\n")


In [ ]:
peaks_sc_m_pw_long <- peaks_sc_m_pw[width(peaks_sc_m_pw) > 9]
peaks_sc_m_pw_long_plus <- peaks_sc_m_pw_long[
  strand(peaks_sc_m_pw_long) == "+"
]
peaks_sc_m_pw_long_plus_curated <- GRanges()

if (length(peaks_sc_m_pw_long_plus) > 0) {
  for (i in seq_along(peaks_sc_m_pw_long_plus)) {
    peak_tmp <- peaks_sc_m_pw_long_plus[i]
    peak_set <- GRanges(
      seqnames = seqnames(peak_tmp),
      ranges = IRanges(
        start = (start(peak_tmp) - 4):(end(peak_tmp) - 4),
        width = 9
      ),
      strand = strand(peak_tmp)
    )
    max_ordered <- order(
      sapply(plus_cov[peak_set - 4], runValue),
      decreasing = TRUE
    )
    peaks_ord <- peak_set[max_ordered]
    peaks_final <- peaks_ord[1]
    peaks_cands <- peaks_ord[-1]
    peaks_cands <- peaks_cands[
      -as.matrix(findOverlaps(peaks_final, peaks_cands))[, 2]
    ]

    while (length(peaks_cands) > 0) {
      peaks_final <- c(peaks_cands[1], peaks_final)
      peaks_cands <- peaks_cands[-1]
      peaks_cands <- peaks_cands[
        -as.matrix(findOverlaps(peaks_final, peaks_cands))[, 2]
      ]
    }
    peaks_sc_m_pw_long_plus_curated <- c(
      peaks_sc_m_pw_long_plus_curated,
      peaks_final
    )
    stopifnot(
      nrow(as.matrix(findOverlaps(peaks_sc_m_pw_long_plus_curated))) ==
        length(peaks_sc_m_pw_long_plus_curated)
    )
    if (i %% 500 == 0) cat("Plus-strand iteration", i, "complete\n")
  }
}


In [ ]:
peaks_sc_m_pw_long_minus <- peaks_sc_m_pw_long[
  strand(peaks_sc_m_pw_long) == "-"
]
peaks_sc_m_pw_long_minus_curated <- GRanges()

if (length(peaks_sc_m_pw_long_minus) > 0) {
  for (i in seq_along(peaks_sc_m_pw_long_minus)) {
    peak_tmp <- peaks_sc_m_pw_long_minus[i]
    peak_set <- GRanges(
      seqnames = seqnames(peak_tmp),
      ranges = IRanges(
        end = (end(peak_tmp) + 4):(start(peak_tmp) + 4),
        width = 9
      ),
      strand = strand(peak_tmp)
    )
    max_ordered <- order(
      sapply(minus_cov[peak_set - 4], runValue),
      decreasing = TRUE
    )
    peaks_ord <- peak_set[max_ordered]
    peaks_final <- peaks_ord[1]
    peaks_cands <- peaks_ord[-1]
    peaks_cands <- peaks_cands[
      -as.matrix(findOverlaps(peaks_final, peaks_cands))[, 2]
    ]

    while (length(peaks_cands) > 0) {
      peaks_final <- c(peaks_cands[1], peaks_final)
      peaks_cands <- peaks_cands[-1]
      peaks_cands <- peaks_cands[
        -as.matrix(findOverlaps(peaks_final, peaks_cands))[, 2]
      ]
    }
    peaks_sc_m_pw_long_minus_curated <- c(
      peaks_sc_m_pw_long_minus_curated,
      peaks_final
    )
    stopifnot(
      nrow(as.matrix(findOverlaps(peaks_sc_m_pw_long_minus_curated))) ==
        length(peaks_sc_m_pw_long_minus_curated)
    )
    if (i %% 500 == 0) cat("Minus-strand iteration", i, "complete\n")
  }
}


In [ ]:
peaks_sl_curated <- c(
  peaks_sc_m_pw_sc,
  peaks_sc_m_pw_long_plus_curated,
  peaks_sc_m_pw_long_minus_curated
)
stopifnot(
  nrow(as.matrix(findOverlaps(peaks_sl_curated))) == length(peaks_sl_curated)
)

num_positions_with_crosslinks <- 1
peaks_sl_curated_plus <- peaks_sl_curated[strand(peaks_sl_curated) == "+"]
peaks_curated_plus <- peaks_sl_curated_plus[
  sapply(lapply(plus_cov[peaks_sl_curated_plus], function(x) x > 0), sum) >=
    num_positions_with_crosslinks
]

peaks_sl_curated_minus <- peaks_sl_curated[strand(peaks_sl_curated) == "-"]
peaks_curated_minus <- peaks_sl_curated_minus[
  sapply(lapply(minus_cov[peaks_sl_curated_minus], function(x) x > 0), sum) >=
    num_positions_with_crosslinks
]
peaks_curated <- c(peaks_curated_plus, peaks_curated_minus)

cat("Number of curated plus-strand peaks:", length(peaks_curated_plus), "\n")
cat("Number of curated minus-strand peaks:", length(peaks_curated_minus), "\n")
cat("Total number of curated peaks:", length(peaks_curated), "\n")
stopifnot(
  nrow(as.matrix(findOverlaps(peaks_curated))) == length(peaks_curated)
)
export(peaks_curated, output_file)
